In [20]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

In [21]:
# загрузка данных
df = pd.read_csv('healthcare_dataset.csv')
df

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55495,eLIZABeTH jaCkSOn,42,Female,O+,Asthma,2020-08-16,Joshua Jarvis,Jones-Thompson,Blue Cross,2650.714952,417,Elective,2020-09-15,Penicillin,Abnormal
55496,KYle pEREz,61,Female,AB-,Obesity,2020-01-23,Taylor Sullivan,Tucker-Moyer,Cigna,31457.797307,316,Elective,2020-02-01,Aspirin,Normal
55497,HEATher WaNG,38,Female,B+,Hypertension,2020-07-13,Joe Jacobs DVM,"and Mahoney Johnson Vasquez,",UnitedHealthcare,27620.764717,347,Urgent,2020-08-10,Ibuprofen,Abnormal
55498,JENniFER JOneS,43,Male,O-,Arthritis,2019-05-25,Kimberly Curry,"Jackson Todd and Castro,",Medicare,32451.092358,321,Elective,2019-05-31,Ibuprofen,Abnormal


## Name

In [22]:
# Предобработка  и анализ признаака Name
# 1. приведем к норм регистру
# 2. избавимся от дубликатов  и пропусков
# 3. выделим в отдельный дата фрейм объекты с более чем 2-х слов(предобработка и анализ) 
#    3.1. выделим префиксы в отдельную категорию
#    3.2. выделим научные звания в отдельную категорию
#    3.3. приведем к норм регистру поколения
#    3.4. так же выделим их в отдельную категорию
#    3.5. join с основным датасетом

In [26]:
# Проверим, как точно называется колонка (Name, name, NAME)
for col in df.columns:
    if col.lower() == "name":
        name_col = col
        break
# Приводим имена к нормальному виду
df["Name_cleaned"] = (
    df[name_col]
    .astype(str)                # преобразуем в строку
    .str.strip()                # убираем пробелы в начале и конце
    .str.replace(r"\s+", " ", regex=True)  # заменяем двойные пробелы одним
    .str.title()                # делаем каждое слово с заглавной буквы
)

In [31]:
df[[name_col,'Name_cleaned']].head(10)

,Name,Name_cleaned
0,Bobby JacksOn,Bobby Jackson
1,LesLie TErRy,Leslie Terry
2,DaNnY sMitH,Danny Smith
3,andrEw waTtS,Andrew Watts
4,adrIENNE bEll,Adrienne Bell
5,EMILY JOHNSOn,Emily Johnson
6,edwArD EDWaRDs,Edward Edwards
7,CHrisTInA MARtinez,Christina Martinez
8,JASmINe aGuIlaR,Jasmine Aguilar
9,ChRISTopher BerG,Christopher Berg


In [35]:
# количество измененных записей
change_name = (df[name_col] !=df['Name_cleaned']).sum()
change_name

np.int64(55467)

In [36]:
# проверим остались ли некоррекртые записи
names_with_wrong_case = df['Name_cleaned'].apply(lambda x: x != x.title() and x.lower() != x.upper()).sum()
names_with_wrong_case

np.int64(0)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55500 entries, 0 to 55499
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Name                55500 non-null  object 
 1   Age                 55500 non-null  int64  
 2   Gender              55500 non-null  object 
 3   Blood Type          55500 non-null  object 
 4   Medical Condition   55500 non-null  object 
 5   Date of Admission   55500 non-null  object 
 6   Doctor              55500 non-null  object 
 7   Hospital            55500 non-null  object 
 8   Insurance Provider  55500 non-null  object 
 9   Billing Amount      55500 non-null  float64
 10  Room Number         55500 non-null  int64  
 11  Admission Type      55500 non-null  object 
 12  Discharge Date      55500 non-null  object 
 13  Medication          55500 non-null  object 
 14  Test Results        55500 non-null  object 
dtypes: float64(1), int64(2), object(12)
memory usage: 6.4

In [12]:
# проверим пропуски
df['Name'].isnull().sum()

np.int64(0)

In [52]:
# проверим дуубликаты
df.duplicated().any()

np.True_

In [53]:
df.duplicated().sum()

np.int64(534)

In [46]:
# проверим дуубликаты
df[df.duplicated(keep=False)]

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results,Name_cleaned
64,Nancy glOVeR,58,Male,A-,Hypertension,2020-05-08,Jennifer Larson,"Khan, and Rodriguez Fischer",Medicare,19183.168885,378,Emergency,2020-06-01,Aspirin,Abnormal,Nancy Glover
107,DAVid higgInS,49,Female,B-,Arthritis,2021-03-05,Erin Henderson MD,"Evans and Hall Schneider,",Medicare,24948.477824,361,Emergency,2021-03-20,Penicillin,Abnormal,David Higgins
148,RoBErt hIGGInS,42,Male,AB-,Asthma,2021-05-06,Scott Davis,"and Ford Lee, Rodriguez",Medicare,13355.782085,451,Elective,2021-05-29,Ibuprofen,Inconclusive,Robert Higgins
154,kevIn HiCKs,66,Male,AB+,Arthritis,2021-06-23,Kelly Murphy,Robinson Inc,Medicare,1897.891727,196,Elective,2021-07-09,Ibuprofen,Abnormal,Kevin Hicks
159,miCHAeL TayloR,29,Male,O-,Asthma,2020-02-27,Erica Mccormick,Donaldson-Frey,Medicare,41939.119937,453,Elective,2020-03-26,Ibuprofen,Normal,Michael Taylor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55461,connOR coMPTon,63,Male,A+,Asthma,2021-08-21,Jonathan Allen,"and Willis Mullins, Bowers",Medicare,1936.702824,375,Emergency,2021-09-16,Paracetamol,Normal,Connor Compton
55462,alYSsA mIlLER,35,Female,A-,Diabetes,2022-06-30,Ryan Price,Shelton-Gallagher,UnitedHealthcare,2210.460898,289,Elective,2022-07-27,Penicillin,Normal,Alyssa Miller
55464,ChRIs huGHeS,35,Female,AB-,Obesity,2024-02-28,Katelyn Perry,Lyons-Hansen,Blue Cross,11889.154513,128,Emergency,2024-03-14,Paracetamol,Abnormal,Chris Hughes
55484,keNNEtH alvarez,80,Male,O+,Cancer,2022-05-05,Andrew Conner,Sons Mayo and,Cigna,45653.802310,114,Elective,2022-05-17,Aspirin,Normal,Kenneth Alvarez


In [45]:
df[df['Name_cleaned']== 'Nancy Glover']

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results,Name_cleaned
64,Nancy glOVeR,58,Male,A-,Hypertension,2020-05-08,Jennifer Larson,"Khan, and Rodriguez Fischer",Medicare,19183.168885,378,Emergency,2020-06-01,Aspirin,Abnormal,Nancy Glover
54972,Nancy glOVeR,58,Male,A-,Hypertension,2020-05-08,Jennifer Larson,"Khan, and Rodriguez Fischer",Medicare,19183.168885,378,Emergency,2020-06-01,Aspirin,Abnormal,Nancy Glover


In [47]:
df_clean = df.drop_duplicates()

In [48]:
df_clean.duplicated().any()

np.False_

In [51]:
df_clean[df_clean.duplicated(keep=False)]

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results,Name_cleaned


In [ ]:
df_clean.d